In [20]:
import csv
from itertools import combinations
import networkx as nx

In [21]:
DATA_FOLDER = "../data/"
CSV_PREDICATE_ACTOR = f"{DATA_FOLDER}predicate_actor.csv"
CSV_EDGES_PREDICATE_NETWORK = f"{DATA_FOLDER}edges_predicate_network.csv"
GEPHI_ACTOR_ACTOR_NETWORK_FILE = f"{DATA_FOLDER}gephi/actor_actor_network.gexf"
GEPHI_ACTOR_STANCE_NETWORK_FILE = f"{DATA_FOLDER}gephi/actor_stance_network.gexf"
LIST_OF_YEARS = ["1996", "1997", "1998", "1999", "2000", "2001", "2002", "2003", "2005", "2006", "2007", "2024"]

In [22]:
def get_actor_with_predicate_cooccurrences_by_sentence_for_year(
    year: str,
) -> dict[tuple[str, str, str], dict[str, int]]:
    sentences = {}
    with open(CSV_PREDICATE_ACTOR, mode="r", encoding="utf-8") as file:
        csvFile = csv.DictReader(file)

        for row in csvFile:
            if row["year"].strip() != year:
                continue

            sentence_id = row["sentence_id"].strip()
            actor = row["actor"].strip()
            predicate_type = row["predicate_type"].strip().lower()

            if not actor or predicate_type not in {"support", "opposition"}:
                continue

            if sentence_id not in sentences:
                sentences[sentence_id] = []

            sentences[sentence_id].append((actor, predicate_type))

    cooccurrences = {}

    for sentence_id, rows in sentences.items():
        actors = sorted({actor for actor, _ in rows})

        if len(actors) < 2:
            continue

        actor_predicates = {}
        for actor, predicate_type in rows:
            if actor not in actor_predicates:
                actor_predicates[actor] = []
            actor_predicates[actor].append(predicate_type)

        for actor1, actor2 in combinations(actors, 2):
            key = (year, actor1, actor2)

            if key not in cooccurrences:
                cooccurrences[key] = {
                    "Positive_count": 0,
                    "Negative_count": 0,
                }

            for p1 in actor_predicates[actor1]:
                for p2 in actor_predicates[actor2]:
                    if p1 == p2:
                        cooccurrences[key]["Positive_count"] += 1
                    else:
                        cooccurrences[key]["Negative_count"] += 1

    return cooccurrences

In [23]:
def build_csv_edges_coocurrence_network_with_predicate(cooccurrences: dict[tuple[str, str, str], dict[str, int]]):
    with open(CSV_EDGES_PREDICATE_NETWORK, mode="a", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        for (year, source, target), counts in sorted(cooccurrences.items()):
            weight = counts["Positive_count"] - counts["Negative_count"]
            writer.writerow([
                source,
                target,
                weight,
                year,
                year,
                counts["Positive_count"],
                counts["Negative_count"],
                "positive" if weight > 0 else "negative" if weight < 0 else "neutral"
            ])

In [24]:
with open(CSV_EDGES_PREDICATE_NETWORK, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(['Source', 'Target', 'Weight', 'Start', 'End', 'Positive_count', 'Negative_count', 'Sign'])

for year in LIST_OF_YEARS:
    print(f"Processing year {year}...")
    foo = get_actor_with_predicate_cooccurrences_by_sentence_for_year(year)
    build_csv_edges_coocurrence_network_with_predicate(foo)
    print(f"Finished processing year {year}.\n")

Processing year 1996...
Finished processing year 1996.

Processing year 1997...
Finished processing year 1997.

Processing year 1998...
Finished processing year 1998.

Processing year 1999...
Finished processing year 1999.

Processing year 2000...
Finished processing year 2000.

Processing year 2001...
Finished processing year 2001.

Processing year 2002...
Finished processing year 2002.

Processing year 2003...
Finished processing year 2003.

Processing year 2005...
Finished processing year 2005.

Processing year 2006...
Finished processing year 2006.

Processing year 2007...
Finished processing year 2007.

Processing year 2024...
Finished processing year 2024.



# Graph actor-actor

In [25]:
A = nx.Graph()

with open(CSV_EDGES_PREDICATE_NETWORK, mode="r", encoding="utf-8") as file:
        csvFile = csv.DictReader(file)
        for row in csvFile:
            A.add_edge(
                row["Source"],
                row["Target"],
                weight=abs(int(row["Weight"])),
                start=row["Start"],
                end=row["End"],
                positive_count=row["Positive_count"],
                negative_count=row["Negative_count"],
                sign=row["Sign"]
            )

nx.write_gexf(A, GEPHI_ACTOR_ACTOR_NETWORK_FILE, encoding="utf-8")

print("Nodes:", A.number_of_nodes())
print("Edges:", A.number_of_edges())
print("Density:", nx.density(A))
print("Connected components:", nx.number_connected_components(A))

Nodes: 64
Edges: 92
Density: 0.04563492063492063
Connected components: 7


# Graph biparti actor-stance

In [26]:
import csv
import networkx as nx

B = nx.Graph()
sentence_counts: dict[int, int] = {}

# list with different actors for each sentence_id
with open(CSV_PREDICATE_ACTOR, mode="r", encoding="utf-8") as file:
    csvFile = csv.DictReader(file)
    for row in csvFile:
        if row["predicate_type"] == "other": continue

        sentence_id = int(row["sentence_id"])
        sentence_counts[sentence_id] = sentence_counts.get(sentence_id, 0) + 1

with open(CSV_PREDICATE_ACTOR, mode="r", encoding="utf-8") as file:
    csvFile = csv.DictReader(file)
    for row in csvFile:
        if row["predicate_type"] == "other": continue

        sentence_id = int(row["sentence_id"])
        if sentence_counts.get(sentence_id, 0) < 2: continue

        actor = row["actor"].strip()
        negotiation_point = row["negotiation_point"].strip()

        B.add_node(actor, bipartite="actor")
        B.add_node(negotiation_point, bipartite="negotiation_point")

        B.add_edge(
            actor,
            negotiation_point,
            year=int(row["year"]),
            sentence_id=sentence_id,
            predicate=row["predicate"],
            predicate_type=row["predicate_type"],
        )

nx.write_gexf(B, GEPHI_ACTOR_STANCE_NETWORK_FILE, encoding="utf-8")

print("Nodes:", B.number_of_nodes())
print("Edges:", B.number_of_edges())
print("Density:", nx.density(B))
print("Connected components:", nx.number_connected_components(B))

Nodes: 889
Edges: 882
Density: 0.0022345179825494785
Connected components: 62
